# 04 Variants: the families with their knobs turned

This notebook drives the variants stage a slice at a time. Everything it
calls lives in `songtaste.variants`, `songtaste.diagnostics` and
`songtaste.explore`; every variant run appends to `results/04-variants.csv`
and is skipped next time, so you can run one family, close the kernel,
come back and run the next. The protocol is the screen (5 folds, 2
repeats) declared in `results/protocol.md`, 2026-10-02; `make variants
FAMILY=knn` does the same from the shell.

Run this kernel from the lab folder with its `.venv` (`uv sync`, `make data`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 60)

from songtaste import evaluate, explore, variants, diagnostics as d
from songtaste.evaluate import PROTOCOL
from songtaste.variants import SCREEN, FAMILIES

X, y = evaluate.training_xy(SCREEN)   # 736 songs, duplicates dropped
len(X), FAMILIES

## The catalogue: what each variant changes against its base

In [ ]:
variants.catalogue()

## The feature screen (from `exploration.ipynb`, with equal capacity and a null)

Singles and pairs share the same tree depth, folds are the protocol's, and the
red line is the biggest gain a shuffled copy of the best feature ever produced:
a pair gain below it is noise. Pass any estimator as `model` to ask the same
question of another method (an `RandomForestClassifier(n_estimators=100)`, say).

In [ ]:
screen = explore.feature_screen(X, y, protocol=SCREEN, null_draws=20)   # ~1 min
explore.plot_feature_screen(screen)
screen["pairs"].head(8)

## One family through the screen

Each call appends to `results/04-variants.csv` and skips variants already there.
Rough cost under the screen: linear a few minutes, knn and svm similar, trees
about 15 minutes (the forests). Change `family` and re-run the cell.

In [ ]:
family = "linear"
rows = variants.run_variants(FAMILIES[family], X, y, SCREEN, path=variants.family_file(family))
variants.summarize(rows, SCREEN).round(3)

## Every family screened so far, and what promotion would send to the full protocol

In [ ]:
all_rows = variants.load_all()   # every 04-variants*.csv in results/, the per-family files included
all_rows = all_rows[all_rows["n_repeats"] == SCREEN.n_repeats]
table = variants.summarize(all_rows, SCREEN).round(3)
display(table)
variants.promoted(all_rows, SCREEN).round(3)

In [ ]:
import seaborn as sns
order = table.sort_values(["family", "accuracy mean"], ascending=[True, False]).index
ax = sns.boxplot(all_rows, x="accuracy", y="variant", order=order, hue="family", dodge=False, showmeans=True)
ax.figure.set_size_inches(8, 0.3 * len(order) + 1)
ax.set(xlabel="accuracy on each of the screen's 10 outer folds", ylabel="")

## Why: validation curves

One hyperparameter swept with the rest at defaults, train and validation
score over the screen's folds. The gap between the two lines is the
variance story; where the validation line peaks is the regularisation story.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
d.plot_validation_curve(d.validation_curve_table("logreg", "clf__C", [1e-3, 1e-2, 1e-1, 1, 10, 100], X, y, SCREEN), ax=axes[0])
d.plot_validation_curve(d.validation_curve_table("knn", "clf__n_neighbors", [1, 3, 5, 9, 15, 25, 41, 61], X, y, SCREEN), ax=axes[1])
d.plot_validation_curve(d.validation_curve_table("svm_rbf", "clf__gamma", [1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1], X, y, SCREEN), ax=axes[2])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
d.plot_validation_curve(d.validation_curve_table("tree", "clf__max_depth", [1, 2, 3, 4, 6, 8, 12, 20], X, y, SCREEN), ax=axes[0])
d.plot_validation_curve(d.validation_curve_table("rf", "clf__min_samples_leaf", [1, 2, 3, 5, 10, 20], X, y, SCREEN), ax=axes[1])
d.plot_validation_curve(d.validation_curve_table("boosting", "clf__learning_rate", [0.01, 0.03, 0.1, 0.3, 1.0], X, y, SCREEN), ax=axes[2])

## Why: learning curves

Does more data help? A validation line still rising at 590 songs says
variance (more data or more averaging would pay); two lines that have met
and flattened say bias (a different model class would). `tuned=True` runs
the inner search at each size and is slow; the defaults tell the shape.

In [ ]:
curves = [d.learning_curve_table(name, X, y, SCREEN) for name in ["logreg", "knn", "svm_rbf", "tree", "rf", "boosting"]]
d.plot_learning_curves(curves)

## Robust importance: permutation importance across methods

Each method fitted on the training part of every screen fold, then the drop in
validation accuracy when one original column is shuffled. Rows are ordered by
mean rank across methods, so the top rows are the features every method leans
on, whatever its inductive bias. `tuned=True` for the tuned pipelines (slower).

In [ ]:
pt = d.permutation_table(["logreg", "lda", "knn", "svm_rbf", "rf", "boosting"], X, y, SCREEN, n_repeats=10)
matrix = d.importance_matrix(pt)
plt.figure(figsize=(8, 6))
d.plot_importance_matrix(matrix)
matrix.round(3)

## Out-of-fold ROC, confusion and the threshold

Every song predicted once by a model that never saw it, with the inner search
on. The threshold table says whether deciding at 0.5 leaves accuracy on the
table (it should not, under the 60/40 balance, but the report should show it).

In [ ]:
oofs = [d.oof_predictions(name, X, y, SCREEN) for name in ["logreg", "rf", "svm_rbf"]]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
d.plot_roc(oofs, ax=axes[0])
d.plot_confusion(oofs[1], ax=axes[1])
d.threshold_table(oofs[1]).iloc[::3].round(3)

## The decision rule under its neighbours

What 03-sweep's choice would be under looser or stricter rules, from the
same 25-split table. The report should state this: the forest is the
choice under the rule fixed in advance, and logistic regression under any
looser one.

In [ ]:
sweep = pd.read_csv(evaluate.RESULTS / "03-sweep.csv")
d.decision_sensitivity(sweep)

## Promotion: a variant that earned the full protocol

Once a family's best variant beats its base under the screen, run it under
the full protocol so it can join 03-sweep's comparison on equal footing.
Twenty-five splits with the inner search, so minutes per variant.

In [ ]:
# promoted_names = list(variants.promoted(all_rows, SCREEN).index)
# full = variants.run_variants(promoted_names, *evaluate.training_xy(PROTOCOL), PROTOCOL)
# variants.summarize(full, PROTOCOL).round(3)